In [1]:
pip install pandas numpy requests

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [48]:
import pandas as pd
import numpy as np
import datetime as dt

path = "cancun_weatherr.xlsx"

# Use the row that contains 'time' as the header.
# This is row 4 (0-based index 3), but we can start with header=3 and adjust if needed.
importedWeather = pd.read_excel(path, header=3)
weather = importedWeather.dropna(subset=["time"]).copy()
weather["time"] = pd.to_datetime(weather["time"])
weather = weather.rename(columns={
    "time": "date",
    "uv_index_max ()": "uvIndexMax",
    "temperature_2m_max (Â°F)": "maxTemp",
    "temperature_2m_min (Â°F)": "minTemp",
    "precipitation_sum (mm)": "precipMM",
    "precipitation_hours (h)": "precipHrs",
    "cloud_cover_mean (%)": "avgCloudCover"
})

def classify(row):
    if (row["precipMM"] > 7) & (row["precipHrs"] > 6):
        return "rainy"
    elif row["avgCloudCover"] > 84:
        return "cloudy"
    else:
        return "clear"

def seasonal_multiplier(date):
    month = date.month
    day = date.day
    
    # High season: Christmas, New Year, Spring Break
    if month in [12, 1]:
        return 1.35
    # Spring Break spike (mid-March)
    if (month == 3) and (10 <= day <= 25):
        return 1.55
    # Summer mid-season
    if month in [6, 7, 8]:
        return 1.15
    # Low season (rainy + hurricane season)
    if month in [9, 10]:
        return 0.85
    return 1.0

def weather_adjustment(row):
    temp_effect = (row["maxTemp"] - 80) * 0.003   # hotter days → slightly higher demand
    rain_effect = -0.04 * (row["precipMM"] - 7)           # heavy rain → lower demand
    cloud_effect = -0.005 * (row["avgCloudCover"] - 84)  # cloudier → slightly lower
    return temp_effect + rain_effect + cloud_effect

def generate_synthetic_prices(df):
    fromEWRFlight = 250   # typical Cancun price floor
    avg5StarHotel = 200    # typical Cancun hotel floor
    flights = []
    hotels = []
    
    for _, row in df.iterrows():
        date = row["date"]

        # Weather Adjustments
        season = seasonal_multiplier(date)
        w_adj = weather_adjustment(row)
        
        # Random noise (keeps it realistic)
        flightNoise = np.random.normal(-5, 12)
        hotelNoise = np.random.normal(-3, 9)
        
        # Final synthetic prices
        flightPrice = fromEWRFlight * season * (1 + w_adj) + flightNoise
        hotelPrice = avg5StarHotel * season * (1 + w_adj) + hotelNoise
        # Prevent unrealistic negatives
        flights.append(max(80, round(flightPrice, 2)))
        hotels.append(max(40, round(hotelPrice, 2)))
    
    df["flightPrice"] = flights
    df["hotelPrice"] = hotels
    return df


weather.drop(columns=["sunrise (iso8601)", "sunset (iso8601)", "wind_speed_10m_max (mp/h)", "wind_gusts_10m_max (mp/h)", "rain_sum (mm)", "showers_sum (mm)", "daylight_duration (s)", "precipitation_probability_max (%)", "visibility_mean (m)"], inplace=True)
weather["genLabel"] = weather.apply(classify, axis=1)

syntheticPrices = generate_synthetic_prices(weather)
syntheticPrices.head()

,date,uvIndexMax,maxTemp,minTemp,precipMM,precipHrs,avgCloudCover,genLabel,flightPrice,hotelPrice
0,2023-04-18,8.95,82.1,67.9,0.0,0,43,clear,369.88,296.29
1,2023-04-19,9.00,86.3,70.9,0.0,0,42,clear,381.99,304.85
2,2023-04-20,8.75,86.3,75.3,0.6,6,50,clear,365.90,292.63
3,2023-04-21,9.05,87.8,76.0,0.0,0,49,clear,349.81,280.43
4,2023-04-22,8.20,87.9,76.6,1.7,7,56,clear,319.13,274.58


In [68]:
from sklearn.ensemble import RandomForestRegressor

df = syntheticPrices.copy()
df["date"] = pd.to_datetime(df["date"])
df["doy"] = df["date"].dt.dayofyear
weatherAvg = df.groupby("doy")[inputColumns].mean().reset_index()

inputColumns = [
    "maxTemp",
    "minTemp",
    "precipMM",
    "precipHrs",
    "avgCloudCover"
]
X = df[inputColumns]
y = df[["flightPrice", "hotelPrice"]]

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)
model.fit(X, y)

resultDates = pd.date_range("2027-01-01", "2027-12-31")
future = pd.DataFrame({"date": resultDates})
future["date"] = pd.to_datetime(future["date"])
future["doy"] = future["date"].dt.dayofyear

# Merge into 2027 dataframe
future = future.merge(weatherAvg, on="doy", how="left")
future["month"] = future["date"].dt.month
future["dayOfWeek"] = future["date"].dt.day_name()

# Predict 2027 prices
futureX = future[inputColumns]
predictions = model.predict(future_X)

future["flightPrice"] = predictions[:, 0]
future["hotelPrice"] = predictions[:, 1]
dropCols = inputColumns + ["doy"]
userFutures = future.drop(columns=dropCols)
userFutures["month"] = userFutures["date"].dt.strftime("%b")

userFutures.head()

,date,month,dayOfWeek,flightPrice,hotelPrice
0,2027-01-01,Jan,Friday,412.879500,332.321567
1,2027-01-02,Jan,Saturday,424.155067,342.923467
2,2027-01-03,Jan,Sunday,432.026533,351.077100
3,2027-01-04,Jan,Monday,410.621433,329.897633
4,2027-01-05,Jan,Tuesday,424.501267,336.011800


In [85]:
def searchTrip(future_df, arrival, departure):
    # Convert to datetime
    arrival = pd.to_datetime(arrival)
    departure = pd.to_datetime(departure)

    if departure <= arrival:
        raise ValueError("Departure date must be after arrival date")

    trip = future_df[(future_df["date"] >= arrival) & (future_df["date"] <= departure)].copy()

    if trip.empty:
        raise ValueError("No data available for the selected date range")

    # --- Flight prices ---
    # 
    oneWayPrice = trip.iloc[0]["flightPrice"]
    roundTripPrice = trip.iloc[0]["flightPrice"] + trip.iloc[-1]["flightPrice"]
    oneWayForTwo = oneWayPrice * 2
    roundTripForTwo = roundTripPrice * 2

    # --- Hotel prices ---
    numNights = (departure - arrival).days

    # Sum hotel prices for each night
    hotelTotal = trip.iloc[:-1]["hotelPrice"].sum()  # exclude departure day
    hotelPerNight = hotelTotal / numNights

    return {
        "arrival": arrival.date(),
        "departure": departure.date(),
        "nights": numNights,

        "One Way, One Person": round(oneWayPrice, 2),
        "Round Trip, One Person": round(roundTripPrice, 2),
        "One Way, Two People": round(oneWayForTwo, 2),
        "Round Trip, Two People": round(roundTripForTwo, 2),

        "Hotel Cost Total": round(hotelTotal, 2),
        "Hotel Cost Per Night": round(hotelPerNight, 2)
    }

def printSummary(results):
    arrival = results["arrival"].strftime("%B %d, %Y")
    departure = results["departure"].strftime("%B %d, %Y")
    
    print(f"Trip Summary")
    print(f"Arrival:   {arrival}")
    print(f"Departure: {departure}")
    print(f"Nights:    {results['nights']}\n")

    print("Flight Prices")
    print(f"• One-way (1 person):     ${results['One Way, One Person']:.2f}")
    print(f"• Round-trip (1 person):  ${results['Round Trip, One Person']:.2f}")
    print(f"• One-way (2 people):     ${results['One Way, Two People']:.2f}")
    print(f"• Round-trip (2 people):  ${results['Round Trip, Two People']:.2f}\n")

    print("Hotel Prices")
    print(f"• Total hotel cost:       ${results['Hotel Cost Total']:.2f}")
    print(f"• Price per night:        ${results['Hotel Cost Per Night']:.2f}")

printSummary(searchTrip(userFutures, "9-01-2027", "9-04-2027"))

Trip Summary
Arrival:   September 01, 2027
Departure: September 04, 2027
Nights:    3

Flight Prices
• One-way (1 person):     $300.28
• Round-trip (1 person):  $619.98
• One-way (2 people):     $600.57
• Round-trip (2 people):  $1239.97

Hotel Prices
• Total hotel cost:       $746.73
• Price per night:        $248.91
